# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# Group-aware out-of-fold attribution faithfulness

**Research question:** Do pathology-pretrained Transformer attribution maps faithfully identify the image patches that drive colorectal histology classification, and how do they compare with CNN Grad-CAM?

This notebook reuses the existing frozen-UNI classifier and RGB CNN. It introduces no new model. Each prediction is generated by a model that did not train or validate on that image's source case. Highlighted regions are interpreted only as contributors to model predictions, not as biologically causal tissue regions.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        print(f'GPU {index}:', torch.cuda.get_device_name(index))

In [ ]:
def locate_project_root():
    return _PUBLICATION_ROOT


PROJECT_ROOT = locate_project_root()
DATASET_DIR = (
    PROJECT_ROOT
    / 'Colorectal Histology MNIST'
    / 'Kather_texture_2016_image_tiles_5000'
    / 'Kather_texture_2016_image_tiles_5000'
)
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'grouped_oof_faithfulness'
FOLD_DIR = ARTIFACT_DIR / 'folds'
CHECKPOINT_DIR = ARTIFACT_DIR / 'checkpoints'
UNI_CHECKPOINT_DIR = CHECKPOINT_DIR / 'uni'
CNN_CHECKPOINT_DIR = CHECKPOINT_DIR / 'cnn'
FEATURE_DIR = ARTIFACT_DIR / 'features'
CLASSIFICATION_DIR = ARTIFACT_DIR / 'classification'
FAITHFULNESS_DIR = ARTIFACT_DIR / 'faithfulness'
STABILITY_DIR = ARTIFACT_DIR / 'stability'
FIGURE_DIR = ARTIFACT_DIR / 'figures'
for directory in (
    FOLD_DIR, UNI_CHECKPOINT_DIR, CNN_CHECKPOINT_DIR, FEATURE_DIR,
    CLASSIFICATION_DIR, FAITHFULNESS_DIR, STABILITY_DIR, FIGURE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)
print('Artifacts:', ARTIFACT_DIR)

In [ ]:
from Methods.BaselineCNN import discover_images, set_seed
from Methods.GroupAwareEvaluation import (
    build_faithfulness_cohort,
    build_grouped_oof_assignments,
    create_main_figures,
    evaluate_cnn_faithfulness,
    evaluate_cnn_stability,
    evaluate_uni_faithfulness,
    evaluate_uni_stability,
    extract_all_uni_features,
    fold_class_group_counts,
    freeze_cohort_manifest,
    paired_method_comparisons,
    run_cnn_oof,
    run_uni_oof,
    save_classification_artifacts,
    validate_oof_assignments,
)
from Methods.UNIAttribution import build_uni_classifier, resolve_uni_transform

REFERENCE_SEED = 41
ALL_SEEDS = (11, 23, 41, 57, 73, 89, 101, 131, 151, 181)
DEVICE = torch.device('cuda:0' if torch.cuda.device_count() > 1 else ('cuda:0' if torch.cuda.is_available() else 'cpu'))
NUM_WORKERS = 4
UNI_IMAGE_BATCH_SIZE = 24
UNI_FEATURE_BATCH_SIZE = 128
CNN_BATCH_SIZE = 64
EPOCHS = 40
LEARNING_RATE = 1e-3
PATIENCE = 8
UNI_MODEL_NAME = 'hf-hub:MahmoodLab/uni'
LOCAL_UNI_ASSETS_DIR = None

# Stage 1 is the required one-seed verification. Turn on the two multi-seed
# flags only after all ten folds complete and the assertions below pass.
RUN_MULTI_SEED_OOF = True
RUN_FAITHFULNESS = True
RUN_STABILITY = True
FORCE_FEATURE_REBUILD = False
FORCE_MODEL_RETRAIN = False
FORCE_COHORT_REBUILD = False
set_seed(REFERENCE_SEED)
print('Device:', DEVICE)

## 1. Predefine leakage-free source-group folds

The ten original `CRC-Prim-HE` cases are treated as source groups. Each case is the held-out test group once, while a second disjoint case is used for validation. The complete fold table is saved before model fitting.

In [ ]:
manifest = discover_images(DATASET_DIR)
class_table = manifest[['class_name', 'label']].drop_duplicates().sort_values('label')
CLASS_NAMES = class_table['class_name'].tolist()
assignments = build_grouped_oof_assignments(manifest, seed=REFERENCE_SEED)
validate_oof_assignments(assignments, expected_image_count=len(manifest))
assignments.to_csv(FOLD_DIR / 'fold_assignments.csv', index=False)
assignments[assignments['split'] == 'test'].to_csv(
    FOLD_DIR / 'oof_test_assignments.csv', index=False
)
fold_counts = fold_class_group_counts(assignments)
fold_counts.to_csv(FOLD_DIR / 'fold_class_source_counts.csv', index=False)
source_class_counts = pd.crosstab(manifest['case_id'], manifest['class_name'])
source_class_counts.to_csv(FOLD_DIR / 'source_by_class_counts.csv')

assert manifest['label'].nunique() == 8
assert assignments.query("split == 'test'")['relative_path'].nunique() == len(manifest)
display(source_class_counts)
display(assignments.groupby('fold').agg(
    test_group=('test_group', 'first'),
    validation_group=('validation_group', 'first'),
))

## 2. Load UNI and cache deterministic features once

The frozen encoder is shared across folds. Only the lightweight classifier is refit in each fold and seed. Sign in with the Hugging Face prompt using an account approved for the gated UNI repository.

In [ ]:
from huggingface_hub import login
login()

uni_model = build_uni_classifier(
    PROJECT_ROOT,
    num_classes=len(CLASS_NAMES),
    device=DEVICE,
    model_name=UNI_MODEL_NAME,
    assets_dir=LOCAL_UNI_ASSETS_DIR,
)
uni_transform, uni_data_config = resolve_uni_transform(uni_model.encoder)
UNI_PREPROCESSING_ID = f"{UNI_MODEL_NAME}|{repr(sorted(uni_data_config.items()))}"
feature_cache = extract_all_uni_features(
    uni_model,
    manifest,
    DEVICE,
    FEATURE_DIR / 'uni_cls_all_images.pt',
    image_transform=uni_transform,
    preprocessing_id=UNI_PREPROCESSING_ID,
    batch_size=UNI_IMAGE_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    overwrite=FORCE_FEATURE_REBUILD,
)
print('UNI feature cache:', tuple(feature_cache['features'].shape))
print('UNI preprocessing:', uni_data_config)

## 3. Grouped out-of-fold classification

Run all ten folds for seed 41 first. The aggregate table must contain exactly 5,000 held-out predictions per model and all eight classes. Enabling `RUN_MULTI_SEED_OOF` trains the same folds for all predefined seeds.

In [ ]:
seeds_to_run = ALL_SEEDS if RUN_MULTI_SEED_OOF else (REFERENCE_SEED,)
uni_predictions, uni_history = run_uni_oof(
    uni_model,
    feature_cache,
    manifest,
    assignments,
    CLASS_NAMES,
    DEVICE,
    UNI_CHECKPOINT_DIR,
    seeds=seeds_to_run,
    batch_size=UNI_FEATURE_BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    force_retrain=FORCE_MODEL_RETRAIN,
)
cnn_predictions, cnn_history = run_cnn_oof(
    assignments,
    CLASS_NAMES,
    DEVICE,
    CNN_CHECKPOINT_DIR,
    seeds=seeds_to_run,
    batch_size=CNN_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    force_retrain=FORCE_MODEL_RETRAIN,
)
oof_predictions = pd.concat((uni_predictions, cnn_predictions), ignore_index=True)
if not uni_history.empty:
    uni_history.to_csv(CLASSIFICATION_DIR / 'uni_training_history.csv', index=False)
if not cnn_history.empty:
    cnn_history.to_csv(CLASSIFICATION_DIR / 'cnn_training_history.csv', index=False)

expected = len(manifest)
prediction_counts = oof_predictions.groupby(['model', 'seed']).size()
assert prediction_counts.eq(expected).all(), prediction_counts
assert oof_predictions.groupby(['model', 'seed'])['class_name'].nunique().eq(8).all()
assert not oof_predictions.duplicated(['model', 'seed', 'relative_path']).any()
classification_results, per_class_results = save_classification_artifacts(
    oof_predictions,
    CLASS_NAMES,
    CLASSIFICATION_DIR,
)
display(classification_results.query("scope == 'aggregate_oof'").style.format(precision=4))
display(per_class_results.groupby(['model', 'class_name'])['accuracy'].agg(['mean', 'std']).style.format(precision=4))

In [ ]:
reference_predictions = oof_predictions[oof_predictions['seed'] == REFERENCE_SEED]
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
for axis, model_name in zip(axes, ('UNI', 'CNN')):
    frame = reference_predictions[reference_predictions['model'] == model_name]
    matrix = pd.crosstab(frame['label'], frame['prediction']).reindex(
        index=range(8), columns=range(8), fill_value=0
    )
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axis)
    axis.set(title=f'{model_name}, aggregate OOF seed {REFERENCE_SEED}',
             xlabel='Predicted', ylabel='True')
    axis.tick_params(axis='x', rotation=55)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'oof_confusion_reference_seed.png', dpi=220, bbox_inches='tight')
plt.show()

## 4. Freeze the class-stratified faithfulness cohort

This selection occurs before any attribution result is loaded or computed. It uses identical images for UNI and CNN, includes jointly correct images from all eight classes, deliberately samples low-confidence correct images, includes errors from either model, and rotates across source cases when possible. The default target is 24 correct and up to 10 incorrect images per class.

In [ ]:
proposed_cohort = build_faithfulness_cohort(
    oof_predictions,
    reference_seed=REFERENCE_SEED,
    correct_per_class=24,
    incorrect_per_class=10,
    low_confidence_fraction=1 / 3,
    seed=2027,
)
cohort = freeze_cohort_manifest(
    proposed_cohort,
    ARTIFACT_DIR / 'faithfulness_cohort_manifest.csv',
    overwrite=FORCE_COHORT_REBUILD,
)
assert cohort.loc[cohort['both_correct'], 'label'].nunique() == 8
display(pd.crosstab(cohort['class_name'], cohort['cohort_stratum']))
display(pd.crosstab(cohort['class_name'], cohort['case_id']))
print('Frozen cohort images:', len(cohort))

## 5. Paired logit- and margin-based faithfulness

For every image, each method is compared with one-patch occlusion. Deletion records the predicted-class logit, true-class logit, target-class logit, and target margin at every step. Incorrect predictions receive separate predicted-class and true-class analyses. Random patch orders are shared across methods within each image and target.

In [ ]:
if RUN_FAITHFULNESS:
    uni_faithfulness, uni_curves = evaluate_uni_faithfulness(
        uni_model,
        cohort,
        UNI_CHECKPOINT_DIR,
        REFERENCE_SEED,
        DEVICE,
        FAITHFULNESS_DIR,
        image_transform=uni_transform,
        random_repeats=20,
        heatmap_limit=32,
    )
    cnn_faithfulness, cnn_curves = evaluate_cnn_faithfulness(
        cohort,
        CNN_CHECKPOINT_DIR,
        REFERENCE_SEED,
        DEVICE,
        FAITHFULNESS_DIR,
        num_classes=len(CLASS_NAMES),
        random_repeats=20,
        heatmap_limit=32,
    )
else:
    uni_faithfulness = pd.read_csv(FAITHFULNESS_DIR / 'uni_faithfulness_metrics.csv')
    cnn_faithfulness = pd.read_csv(FAITHFULNESS_DIR / 'cnn_faithfulness_metrics.csv')
    uni_curves = pd.read_csv(FAITHFULNESS_DIR / 'uni_deletion_curves.csv')
    cnn_curves = pd.read_csv(FAITHFULNESS_DIR / 'cnn_deletion_curves.csv')

faithfulness_metrics = pd.concat((uni_faithfulness, cnn_faithfulness), ignore_index=True)
deletion_curves = pd.concat((uni_curves, cnn_curves), ignore_index=True)
faithfulness_metrics.to_csv(FAITHFULNESS_DIR / 'all_faithfulness_metrics.csv', index=False)
deletion_curves.to_csv(FAITHFULNESS_DIR / 'all_deletion_curves.csv', index=False)
faithfulness_summary = (
    faithfulness_metrics.groupby(
        ['model', 'method', 'target_role', 'class_name', 'correct'], dropna=False
    )
    .agg(images=('cohort_id', 'nunique'),
         mean_spearman=('attribution_occlusion_spearman', 'mean'),
         mean_top_logit_auc=('top_target_logit_auc', 'mean'),
         mean_random_logit_auc=('random_target_logit_auc', 'mean'),
         mean_bottom_logit_auc=('bottom_target_logit_auc', 'mean'),
         mean_top_minus_random_auc=('top_minus_random_target_logit_auc', 'mean'))
    .reset_index()
)
faithfulness_summary.to_csv(FAITHFULNESS_DIR / 'aggregate_faithfulness_summary.csv', index=False)
deletion_curve_summary = (
    deletion_curves.groupby(
        ['model', 'method', 'target_role', 'strategy', 'fraction_removed'], dropna=False
    )[['target_class_logit_drop', 'true_class_logit_drop', 'target_margin_drop']]
    .agg(['mean', 'sem']).reset_index()
)
deletion_curve_summary.columns = [
    '_'.join(str(part) for part in column if part).rstrip('_')
    if isinstance(column, tuple) else column
    for column in deletion_curve_summary.columns
]
deletion_curve_summary.to_csv(FAITHFULNESS_DIR / 'aggregate_deletion_curves.csv', index=False)
display(
    faithfulness_metrics.groupby(['model', 'method', 'target_role'])
    .agg(images=('cohort_id', 'nunique'),
         mean_spearman=('attribution_occlusion_spearman', 'mean'),
         mean_top_minus_random_auc=('top_minus_random_target_logit_auc', 'mean'))
    .style.format(precision=4)
)

## 6. Paired statistical comparison

The primary direct comparison is UNI gradient-weighted rollout versus CNN Grad-CAM for the same image and the same true-class target. The secondary predicted-class comparison includes only images for which both models predicted the same class. Tests are paired Wilcoxon signed-rank tests with Holm correction. Confidence intervals are reported from both image-level paired bootstrap and source-group-first hierarchical bootstrap.

In [ ]:
paired_tests, paired_values = paired_method_comparisons(
    faithfulness_metrics,
    bootstrap_iterations=5000,
    confidence=0.95,
    random_seed=2027,
)
paired_tests.to_csv(FAITHFULNESS_DIR / 'paired_wilcoxon_bootstrap_results.csv', index=False)
paired_values.to_csv(FAITHFULNESS_DIR / 'paired_uni_cnn_values.csv', index=False)
display(
    paired_tests.query("comparison_target == 'true_class' and subgroup == 'all'")
    .style.format(precision=4)
)

## 7. Cross-seed prediction and attribution stability

Run this section only after `RUN_MULTI_SEED_OOF=True` has produced all fold checkpoints. Predicted-class maps are compared only for seed pairs that predict the same class. A separate true-class-targeted map is computed for every seed so differing predictions can still be compared without mixing target classes.

In [ ]:
if RUN_STABILITY:
    missing_uni = [
        (seed, fold) for seed in ALL_SEEDS for fold in assignments['fold'].unique()
        if not (UNI_CHECKPOINT_DIR / f'seed_{seed}' / f'fold_{fold}.pt').is_file()
    ]
    missing_cnn = [
        (seed, fold) for seed in ALL_SEEDS for fold in assignments['fold'].unique()
        if not (CNN_CHECKPOINT_DIR / f'seed_{seed}' / f'fold_{fold}.pt').is_file()
    ]
    if missing_uni or missing_cnn:
        raise RuntimeError('Complete multi-seed OOF training before stability analysis')
    _, _, uni_stability = evaluate_uni_stability(
        uni_model, cohort, UNI_CHECKPOINT_DIR, ALL_SEEDS, DEVICE,
        STABILITY_DIR, image_transform=uni_transform,
    )
    _, _, cnn_stability = evaluate_cnn_stability(
        cohort, CNN_CHECKPOINT_DIR, ALL_SEEDS, DEVICE, STABILITY_DIR,
        num_classes=len(CLASS_NAMES),
    )
    stability_metrics = pd.concat((uni_stability, cnn_stability), ignore_index=True)
    stability_metrics.to_csv(STABILITY_DIR / 'all_stability_per_image.csv', index=False)
    stability_summary = (
        stability_metrics.groupby(['model', 'class_name', 'cohort_stratum'])
        [['correct_rate_across_seeds', 'pairwise_prediction_agreement',
          'mean_predicted_class_spearman_same_prediction',
          'mean_common_true_class_spearman']]
        .agg(['mean', 'sem']).reset_index()
    )
    stability_summary.to_csv(STABILITY_DIR / 'aggregate_stability_summary.csv', index=False)
else:
    stability_path = STABILITY_DIR / 'all_stability_per_image.csv'
    stability_metrics = pd.read_csv(stability_path) if stability_path.is_file() else pd.DataFrame()
if not stability_metrics.empty:
    display(
        stability_metrics.groupby(['model', 'class_name', 'cohort_stratum'])
        [['pairwise_prediction_agreement',
          'mean_predicted_class_spearman_same_prediction',
          'mean_common_true_class_spearman']]
        .mean().style.format(precision=4)
    )

## 8. Main figures and interpretation guardrails

In [ ]:
figure_paths = create_main_figures(
    classification_results,
    faithfulness_metrics,
    deletion_curves,
    stability_metrics,
    CLASS_NAMES,
    FIGURE_DIR,
)
for path in figure_paths:
    print(path)

print('\nInterpretation rules:')
print('- Classification and faithfulness are reported separately.')
print('- Positive attribution-occlusion correlation and stronger top-than-random deletion support faithfulness.')
print('- Raw attention and ordinary rollout are weak baselines, not explanations by themselves.')
print('- Do not compare predicted-class maps across seeds when the predicted classes differ.')
print('- Do not call highlighted patches biologically causal without external pathology annotations.')
print('- Do not claim general Grad-CAM superiority unless the paired class-complete results support it.')